In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

X,y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model =keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(8, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1,activation="sigmoid"),
])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 16)             │           496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 641 (2.50 KB)

 Trainable params: 641 (2.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss=keras.losses.BinaryCrossentropy(),
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=1, batch_size=4,verbose=2,callbacks=[early_stopping])

91/91 - 4s - 44ms/step - accuracy: 0.4148 - loss: 0.8161 - val_accuracy: 0.4066 - val_loss: 0.7332


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader,random_split

In [ ]:
device =torch.device("cuda"if  torch.cuda.is_available() else "cpu")

In [ ]:
X_train_tensor = torch.tensor(X_train,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test,dtype=torch.float32).unsqueeze(1)

In [ ]:
full_ds = TensorDataset(X_train_tensor,y_train_tensor)
train_ds,val_ds = random_split(full_ds,[int(0.8*len(full_ds)),len(full_ds)-int(0.8*len(full_ds))])

train_loader = DataLoader(train_ds,batch_size=32,shuffle=True)
val_loader = DataLoader(val_ds,batch_size=32)

In [ ]:
class ANN(nn.Module):
  def __init__ (self,n_features):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_features,16),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(16,1),
        nn.Dropout(0.2),
    )

  def forward(self,x):
    return self.net(x)

model = ANN(X_train.shape[1]).to(device)
print(model)

ANN(
  (net): Sequential(
    (0): Linear(in_features=30, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=16, out_features=1, bias=True)
    (4): Dropout(p=0.2, inplace=False)
  )
)


In [ ]:
criterion = nn.BCEWithLogitsLoss() # sigmoid + binary cross entropy
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
def accuracy(logits,targets):
  preds = (torch.sigmoid(logits)>0.5).float()
  return (preds==targets).float().mean()

In [ ]:
EPOCHS =20

for epoch in range(1,EPOCHS+1):
  model.train()
  train_loss = 0
  train_acc = 0
  for batch_idx,(data,targets) in enumerate(train_loader):
    data,targets = data.to(device),targets.to(device)
    optimizer.zero_grad()
    logits = model(data)
    loss= criterion(logits,targets)
    loss.backward()
    optimizer.step()
    train_loss+=loss.item()*data.size(0)
    train_acc+=accuracy(logits,targets)

In [ ]:
model.eval()
val_loss,val_correct = 0.0,0.0
with torch.no_grad():
  for data,targets in val_loader:
    data,targets = data.to(device),targets.to(device)
    logits = model(data)
    val_loss += criterion(logits,targets).item()*data.size(0)
    val_correct += accuracy(logits,targets)*data.size(0)
val_loss /= len(val_loader.dataset)
val_acc =val_correct/len(val_loader.dataset)
print(f"Epoch: {epoch}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")


Epoch: 20, Validation Loss: 0.1655, Validation Accuracy: 0.9780


In [ ]:
model.eval()
with torch.no_grad():
  test_logits = model(X_test_tensor.to(device))
  test_loss = criterion(test_logits,y_test_tensor.to(device))
  test_acc = accuracy(test_logits,y_test_tensor.to(device))
print(f"\n Test kaybı: {test_loss:.4f} | Test doğruluğu {test_acc:.4f}")


 Test kaybı: 0.1092 | Test doğruluğu 0.9912


In [ ]:
t= np.arange(2000)
series = np.sin(0.05*t)+0.5*np.sin(0.011*t)+np.random.normal(0,0.1,len(t))
series = series.astype("float32")

In [ ]:
WİNDOW = 30

def make_windows(data,window):
  X,y =[] ,[]
  for i in range(len(data)-window):
    X.append(data[i:i+window])
    y.append(data[i+window])
  return np.array(X),np.array(y)

X,y = make_windows(series,WİNDOW)

In [ ]:
split = int(0.8*len(X))
X_train,X_test = X[:split],X[split:]
y_train,y_test = y[:split],y[split:]
print("X_train",X_train.shape)
print("X_test",X_test.shape)
print("y_train",y_train.shape)
print("y_test",y_test.shape)

X_train (1576, 30)
X_test (394, 30)
y_train (1576,)
y_test (394,)


In [ ]:
def build_model(cell ="lstm",units =32):
  rnn_layer = layers.LSTM(units) if cell == "lstm" else layers.GRU(units)
  model = keras.Sequential([
      layers.Input(shape=(X_train.shape[1],1)),
      rnn_layer,
      layers.Dense(1)
  ])
  model.compile(optimizer=keras.optimizers.Adam(1e-3),loss="mse",metrics =["mae"])
  return model

In [ ]:
results = {}

for cell in ["lstm","gru"]:
  model = build_model(cell)
  model.summary()
  model.fit(
      X_train,
      y_train,
      epochs =20,
      batch_size =32,
      validation_split =0.1,
      verbose =2,
  )

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,385 (17.13 KB)

 Trainable params: 4,385 (17.13 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 32)             │         3,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,393 (13.25 KB)

 Trainable params: 3,393 (13.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
45/45 - 3s - 61ms/step - loss: 0.4151 - mae: 0.5124 - val_loss: 0.0519 - val_mae: 0.1943
Epoch 2/20
45/45 - 1s - 16ms/step - loss: 0.0337 - mae: 0.1513 - val_loss: 0.0332 - val_mae: 0.1524
Epoch 3/20
45/45 - 1s - 15ms/step - loss: 0.0263 - mae: 0.1331 - val_loss: 0.0279 - val_mae: 0.1373
Epoch 4/20
45/45 - 1s - 15ms/step - loss: 0.0232 - mae: 0.1237 - val_loss: 0.0249 - val_mae: 0.1289
Epoch 5/20
45/45 - 1s - 28ms/step - loss: 0.0209 - mae: 0.1166 - val_loss: 0.0228 - val_mae: 0.1235
Epoch 6/20
45/45 - 1s - 15ms/step - loss: 0.0191 - mae: 0.1109 - val_loss: 0.0212 - val_mae: 0.1198
Epoch 7/20
45/45 - 1s - 14ms/step - loss: 0.0177 - mae: 0.1063 - val_loss: 0.0199 - val_mae: 0.1165
Epoch 8/20
45/45 - 1s - 15ms/step - loss: 0.0165 - mae: 0.1024 - val_loss: 0.0187 - val_mae: 0.1135
Epoch 9/20
45/45 - 1s - 19ms/step - loss: 0.0155 - mae: 0.0989 - val_loss: 0.0175 - val_mae: 0.1103
Epoch 10/20
45/45 - 1s - 27ms/step - loss: 0.0147 - mae: 0.0960 - val_loss: 0.0165 - val_mae: 0.1073

In [ ]:
mse,mae = model.evaluate(X_test,y_test)
print("MSE",mse)
print("MAE",mae)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0125 - mae: 0.0876
MSE 0.012495309114456177
MAE 0.08764965087175369


In [ ]:
VOCAB_SİZE = 10000
MAX_LEN = 200
EMBED_DIM = 128
LSTM =64

In [ ]:
(X_train,y_train),(X_test,y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SİZE)
print(f"Eğitim:{len(X_train)} yorum | Test :{len(X_test)} yorum")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Eğitim:25000 yorum | Test :25000 yorum


In [ ]:
word_index = keras.datasets.imdb.get_word_index()
index_word = {index+3:word for word,index in word_index.items()}
index_word.update({0:"<PAD>",1:"<START>",2:"<OOV>"})
print("\n Örnek yorum:","".join(index_word.get(i,"?")for i in X_train[0][:40]),"....")
print("Etiket:","olumlu" if y_train[0]==1 else "olumsuz")


 Örnek yorum: <START>thisfilmwasjustbrilliantcastinglocationscenerystorydirectioneveryone'sreallysuitedtheparttheyplayedandyoucouldjustimaginebeingthererobert<OOV>isanamazingactorandnowthesamebeingdirector<OOV>fathercame ....
Etiket: olumlu


In [ ]:
X_train = keras.utils.pad_sequences(X_train,maxlen=MAX_LEN,padding="post",truncating= "post")
X_test = keras.utils.pad_sequences(X_test,maxlen=MAX_LEN,padding="post",truncating= "post")

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=VOCAB_SİZE,output_dim=EMBED_DIM, mask_zero=True),
    layers.LSTM(LSTM),
    layers.Dropout(0.2),
    layers.Dense(1,activation="sigmoid"),
    ])





In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss",patience=2,restore_best_weights=True)
model.compile(optimizer=keras.optimizers.Adam(1e-3),loss="binary_crossentropy",metrics=["accuracy"])
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,329,473 (5.07 MB)

 Trainable params: 1,329,473 (5.07 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import re
def encode_text(text):
    """Ham İngilizce metni modelin beklediği indeks dizisine çevirir."""
    words = re.sub(r"[^a-z0-9' ]", " ", text.lower()).split()
    ids = [1]  # <START>
    for w in words:
        idx = word_index.get(w)
        # Kelime sözlükte yoksa veya ilk VOCAB_SIZE içinde değilse -> <OOV>
        ids.append(idx + 3 if idx is not None and idx + 3 < VOCAB_SİZE else 2)
    return keras.utils.pad_sequences([ids], maxlen=MAX_LEN, padding="post", truncating="post")

examples = [
    "This movie was absolutely wonderful, the acting was brilliant and I loved every minute.",
    "What a waste of time. The plot was boring and the characters were terrible.",
    "It was okay, not great but not bad either.",
]

print("\n--- Yeni yorumlar üzerinde tahmin ---")
for text in examples:
    prob = float(model.predict(encode_text(text), verbose=0)[0, 0])
    label = "OLUMLU" if prob > 0.5 else "OLUMSUZ"
    print(f"[{label} | {prob:.3f}] {text}")




--- Yeni yorumlar üzerinde tahmin ---
[OLUMLU | 0.502] This movie was absolutely wonderful, the acting was brilliant and I loved every minute.
[OLUMLU | 0.505] What a waste of time. The plot was boring and the characters were terrible.
[OLUMSUZ | 0.493] It was okay, not great but not bad either.
